In [1]:
# 📊 Prediction Confidence & Uncertainty Range

## Mumbai House Price Prediction Project

This notebook estimates model-based prediction uncertainty for Mumbai house
price predictions using the individual trees of the trained Random Forest model.

### Objectives

- Load the trained Random Forest model
- Load the saved preprocessing pipeline
- Load the original model input features
- Generate individual tree predictions
- Calculate prediction spread
- Estimate lower and upper price ranges
- Calculate uncertainty percentage
- Assign confidence levels
- Analyze uncertainty across Mumbai localities
- Export prediction confidence results

### Important Note

The generated price range is a model-based uncertainty estimate.

It is NOT a formal statistical confidence interval.

SyntaxError: invalid syntax (1211812800.py, line 5)

In [2]:
## 🔬 Method Used

The trained Random Forest model contains multiple decision trees.

Each tree generates an individual prediction for the same property.

The variation between these individual tree predictions is used to estimate
model uncertainty.

For each property we calculate:

- Mean predicted price
- Standard deviation of tree predictions
- Lower estimated price
- Upper estimated price
- Prediction range
- Uncertainty percentage
- Confidence level

A smaller prediction spread indicates greater agreement between the trees,
while a larger spread indicates greater model uncertainty.

SyntaxError: invalid syntax (3998978077.py, line 3)

In [3]:
import os
import glob
import numpy as np
import pandas as pd
import joblib

from IPython.display import display

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


In [4]:
PROJECT_DIR = r"C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction"

PROCESSED_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "processed"
)

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "outputs"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("Project Directory:")
print(PROJECT_DIR)

print()
print("Processed Directory:")
print(PROCESSED_DIR)

print()
print("Model Directory:")
print(MODEL_DIR)

print()
print("Output Directory:")
print(OUTPUT_DIR)

Project Directory:
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction

Processed Directory:
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\data\processed

Model Directory:
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models

Output Directory:
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\outputs


In [5]:
# Clear variables from previous failed runs

variables_to_clear = [
    "df",
    "training_df",
    "X_raw",
    "X_processed",
    "X_sample",
    "sample_df",
    "tree_predictions",
    "mean_predictions",
    "std_predictions",
    "lower_predictions",
    "upper_predictions",
    "confidence_df",
    "locality_uncertainty",
    "highest_uncertainty_localities",
    "lowest_uncertainty_localities"
]

for variable_name in variables_to_clear:

    if variable_name in globals():

        del globals()[variable_name]

print("✅ Old confidence-analysis variables cleared.")

✅ Old confidence-analysis variables cleared.


In [6]:
model_files = []

for root, dirs, files in os.walk(MODEL_DIR):

    for file in files:

        if file.lower().endswith(
            (".pkl", ".joblib")
        ):

            model_files.append(
                os.path.join(root, file)
            )

print("Model files found:")
print()

for file_path in model_files:

    print(file_path)

Model files found:

C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models\best_model_info.joblib
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models\best_tuned_model.joblib
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models\gradient_boosting_log_model.joblib
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models\gradient_boosting_model.joblib
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models\isolation_forest_model.joblib
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models\lasso_model.joblib
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models\linear_log_model.joblib
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models\linear_model.joblib
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models\preprocessor.joblib
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-pr

In [7]:
preferred_model_names = [
    "model.pkl",
    "model.joblib",
    "random_forest_model.pkl",
    "random_forest_model.joblib",
    "best_model.pkl",
    "best_model.joblib",
    "tuned_model.pkl",
    "tuned_model.joblib"
]

model_path = None

for preferred_name in preferred_model_names:

    for file_path in model_files:

        if os.path.basename(file_path).lower() == preferred_name.lower():

            model_path = file_path
            break

    if model_path is not None:
        break


if model_path is None and len(model_files) > 0:

    model_path = model_files[0]


if model_path is not None:

    print("✅ Model selected:")
    print(model_path)

else:

    raise FileNotFoundError(
        "No model .pkl or .joblib file found."
    )

✅ Model selected:
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models\random_forest_model.joblib


In [8]:
model = joblib.load(
    model_path
)

print("✅ Model loaded successfully.")
print()
print("Model type:")
print(type(model))

✅ Model loaded successfully.

Model type:
<class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [9]:
if not hasattr(model, "estimators_"):

    raise ValueError(
        "The loaded model does not contain individual tree estimators."
    )

print("✅ Random Forest tree estimators detected.")
print()
print(
    "Number of trees:",
    len(model.estimators_)
)

✅ Random Forest tree estimators detected.

Number of trees: 150


In [10]:
preprocessor_files = []

for root, dirs, files in os.walk(MODEL_DIR):

    for file in files:

        if (
            "preprocessor" in file.lower()
            and file.lower().endswith(
                (".pkl", ".joblib")
            )
        ):

            preprocessor_files.append(
                os.path.join(root, file)
            )

print("Preprocessor files found:")
print()

for file_path in preprocessor_files:

    print(file_path)

Preprocessor files found:

C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models\preprocessor.joblib


In [11]:
preferred_preprocessor_names = [
    "preprocessor.pkl",
    "preprocessor.joblib",
    "preprocessor_pipeline.pkl",
    "preprocessor_pipeline.joblib"
]

preprocessor_path = None

for preferred_name in preferred_preprocessor_names:

    for file_path in preprocessor_files:

        if os.path.basename(file_path).lower() == preferred_name.lower():

            preprocessor_path = file_path
            break

    if preprocessor_path is not None:
        break


if preprocessor_path is None and len(preprocessor_files) > 0:

    preprocessor_path = preprocessor_files[0]


if preprocessor_path is not None:

    print("✅ Preprocessor selected:")
    print(preprocessor_path)

else:

    raise FileNotFoundError(
        "No preprocessor .pkl or .joblib file found."
    )

✅ Preprocessor selected:
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\models\preprocessor.joblib


In [12]:
preprocessor = joblib.load(
    preprocessor_path
)

print("✅ Preprocessor loaded successfully.")
print()
print("Preprocessor type:")
print(type(preprocessor))

✅ Preprocessor loaded successfully.

Preprocessor type:
<class 'sklearn.compose._column_transformer.ColumnTransformer'>


In [14]:
if hasattr(
    preprocessor,
    "feature_names_in_"
):

    expected_features = list(
        preprocessor.feature_names_in_
    )

    print(
        "Number of expected raw features:",
        len(expected_features)
    )

    print()

    for i, feature in enumerate(
        expected_features,
        start=1
    ):

        print(
            f"{i}. {feature}"
        )

else:

    raise ValueError(
        "Preprocessor does not expose feature_names_in_."
    )

Number of expected raw features: 15

1. area
2. locality
3. city
4. property_type
5. bedroom_num
6. bathroom_num
7. balcony_num
8. furnished
9. age
10. total_floors
11. latitude
12. longitude
13. area_per_bedroom
14. bathroom_per_bedroom
15. age_group


In [15]:
expected_feature_set = set(
    expected_features
)

print()
print("Expected model input features:")
print(expected_features)

print()
print(
    "Total expected features:",
    len(expected_features)
)


Expected model input features:
['area', 'locality', 'city', 'property_type', 'bedroom_num', 'bathroom_num', 'balcony_num', 'furnished', 'age', 'total_floors', 'latitude', 'longitude', 'area_per_bedroom', 'bathroom_per_bedroom', 'age_group']

Total expected features: 15


In [16]:
CLEANED_DATA_PATH = os.path.join(
    PROCESSED_DIR,
    "mumbai_house_price_cleaned.csv"
)

print("Looking for:")
print(CLEANED_DATA_PATH)

if not os.path.exists(
    CLEANED_DATA_PATH
):

    raise FileNotFoundError(
        "mumbai_house_price_cleaned.csv was not found."
    )

print()
print("✅ Correct model-input dataset found.")

Looking for:
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\data\processed\mumbai_house_price_cleaned.csv

✅ Correct model-input dataset found.


In [17]:
training_df = pd.read_csv(
    CLEANED_DATA_PATH
)

print("✅ Training feature dataset loaded.")
print()
print("Dataset shape:")
print(training_df.shape)

✅ Training feature dataset loaded.

Dataset shape:
(51767, 14)


In [18]:
display(
    training_df.head()
)

,price,area,price_per_sqft,locality,city,property_type,bedroom_num,bathroom_num,balcony_num,furnished,age,total_floors,latitude,longitude
0,6600283,757,8719.000000,Kalyan,Mumbai,Apartment,2,2,0,Unfurnished,0,1,19.244410,73.123253
1,6169841,652,9462.946319,Kalyan,Mumbai,Apartment,2,2,0,Unfurnished,0,1,19.257294,73.148872
2,4599936,396,11616.000000,Dombivali,Mumbai,Apartment,1,1,0,Unfurnished,0,1,19.209026,73.081276
3,51980000,1130,46000.000000,Ville Parle,Mumbai,Apartment,3,3,0,Unfurnished,0,1,19.097841,72.851158
4,3915000,435,9000.000000,Nala Sopara,Mumbai,Apartment,1,1,0,Unfurnished,0,1,19.420601,72.809319


In [19]:
print("Available columns:")
print()

for i, column in enumerate(
    training_df.columns,
    start=1
):

    print(
        f"{i}. {column}"
    )

Available columns:

1. price
2. area
3. price_per_sqft
4. locality
5. city
6. property_type
7. bedroom_num
8. bathroom_num
9. balcony_num
10. furnished
11. age
12. total_floors
13. latitude
14. longitude


In [21]:
# ============================================================
# CELL 19 — CREATE MISSING ENGINEERED FEATURES
# ============================================================

training_df = training_df.copy()

print("Creating required engineered features...")

# ------------------------------------------------------------
# 1. area_per_bedroom
# ------------------------------------------------------------
training_df["area_per_bedroom"] = (
    training_df["area"] /
    training_df["bedroom_num"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 2. bathroom_per_bedroom
# ------------------------------------------------------------
training_df["bathroom_per_bedroom"] = (
    training_df["bathroom_num"] /
    training_df["bedroom_num"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 3. age_group
# ------------------------------------------------------------
def create_age_group(age):
    if pd.isna(age):
        return "Unknown"
    elif age <= 5:
        return "New"
    elif age <= 10:
        return "Recent"
    elif age <= 20:
        return "Moderate"
    else:
        return "Old"

training_df["age_group"] = training_df["age"].apply(create_age_group)

# ------------------------------------------------------------
# Handle missing values created by division
# ------------------------------------------------------------
training_df["area_per_bedroom"] = (
    training_df["area_per_bedroom"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

training_df["bathroom_per_bedroom"] = (
    training_df["bathroom_per_bedroom"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# ------------------------------------------------------------
# Check required features again
# ------------------------------------------------------------
missing_features = [
    feature
    for feature in expected_features
    if feature not in training_df.columns
]

print("\nRequired features:", len(expected_features))
print("Missing features:", len(missing_features))

if len(missing_features) == 0:
    print("✅ ALL REQUIRED MODEL FEATURES ARE NOW AVAILABLE.")
else:
    print("❌ Still missing:")
    for feature in missing_features:
        print("-", feature)
    raise ValueError("Required model features are still missing.")

# ------------------------------------------------------------
# Display created features
# ------------------------------------------------------------
print("\nCreated engineered features:")

display(
    training_df[
        [
            "area_per_bedroom",
            "bathroom_per_bedroom",
            "age_group"
        ]
    ].head()
)

print("\nTraining data shape:", training_df.shape)

Creating required engineered features...

Required features: 15
Missing features: 0
✅ ALL REQUIRED MODEL FEATURES ARE NOW AVAILABLE.

Created engineered features:


,area_per_bedroom,bathroom_per_bedroom,age_group
0,378.500000,1.0,New
1,326.000000,1.0,New
2,396.000000,1.0,New
3,376.666667,1.0,New
4,435.000000,1.0,New



Training data shape: (51767, 17)


In [23]:
X_raw = training_df[
    expected_features
].copy()

print("✅ Raw model input created.")
print()
print("X_raw shape:")
print(X_raw.shape)

print()
print("X_raw columns:")
print(X_raw.columns.tolist())

✅ Raw model input created.

X_raw shape:
(51767, 15)

X_raw columns:
['area', 'locality', 'city', 'property_type', 'bedroom_num', 'bathroom_num', 'balcony_num', 'furnished', 'age', 'total_floors', 'latitude', 'longitude', 'area_per_bedroom', 'bathroom_per_bedroom', 'age_group']


In [24]:
missing_values = X_raw.isnull().sum()

missing_values = (
    missing_values[
        missing_values > 0
    ]
    .sort_values(
        ascending=False
    )
)

if len(missing_values) == 0:

    print(
        "✅ No missing values in model input."
    )

else:

    print(
        "⚠️ Missing values found:"
    )

    display(
        missing_values
    )

✅ No missing values in model input.


In [25]:
print("Model input data types:")
print()

display(
    X_raw.dtypes.to_frame(
        "dtype"
    )
)

Model input data types:



,dtype
area,int64
locality,str
city,str
property_type,str
bedroom_num,int64
bathroom_num,int64
balcony_num,int64
furnished,str
age,int64
total_floors,int64


In [26]:
try:

    X_processed = preprocessor.transform(
        X_raw
    )

    print(
        "✅ Data transformed successfully."
    )

    print()
    print(
        "Processed data shape:"
    )

    print(
        X_processed.shape
    )

except Exception as e:

    print(
        "❌ Preprocessing failed."
    )

    print()
    print(
        "Error:"
    )

    print(e)

    raise

✅ Data transformed successfully.

Processed data shape:
(51767, 112)


In [27]:
if not hasattr(
    model,
    "n_features_in_"
):

    raise ValueError(
        "Model does not expose n_features_in_."
    )

model_expected_features = (
    model.n_features_in_
)

processed_feature_count = (
    X_processed.shape[1]
)

print(
    "Model expects processed features:",
    model_expected_features
)

print(
    "Processed data contains:",
    processed_feature_count
)

print()

if (
    model_expected_features
    == processed_feature_count
):

    print(
        "✅ Processed feature count matches model."
    )

else:

    raise ValueError(
        "Processed feature count does not match model."
    )

Model expects processed features: 112
Processed data contains: 112

✅ Processed feature count matches model.


In [28]:
## 🌲 Random Forest Tree Predictions

The Random Forest contains many individual decision trees.

Instead of using only the final Random Forest prediction, we obtain the
prediction generated by every individual tree.

The variation between these tree predictions will be used as a model-based
uncertainty estimate.

SyntaxError: invalid syntax (597196469.py, line 3)

In [29]:
def get_tree_predictions(
    model,
    X
):

    if not hasattr(
        model,
        "estimators_"
    ):

        raise ValueError(
            "Model does not contain individual tree estimators."
        )

    predictions = []

    for tree in model.estimators_:

        tree_prediction = tree.predict(
            X
        )

        predictions.append(
            tree_prediction
        )

    predictions = np.asarray(
        predictions
    )

    return predictions


print(
    "✅ Tree prediction function created."
)

✅ Tree prediction function created.


In [30]:
MAX_SAMPLE_SIZE = 1000

sample_size = min(
    MAX_SAMPLE_SIZE,
    X_processed.shape[0]
)

random_state = np.random.RandomState(
    42
)

sample_indices = random_state.choice(
    X_processed.shape[0],
    size=sample_size,
    replace=False
)

X_sample = X_processed[
    sample_indices
]

sample_df = training_df.iloc[
    sample_indices
].copy()

print(
    "✅ Memory-safe sample created."
)

print()
print(
    "Sample size:",
    sample_size
)

print(
    "X_sample shape:",
    X_sample.shape
)

print(
    "sample_df shape:",
    sample_df.shape
)

✅ Memory-safe sample created.

Sample size: 1000
X_sample shape: (1000, 112)
sample_df shape: (1000, 17)


In [31]:
tree_predictions = get_tree_predictions(
    model,
    X_sample
)

print(
    "✅ Tree predictions generated."
)

print()
print(
    "Tree prediction array shape:"
)

print(
    tree_predictions.shape
)

✅ Tree predictions generated.

Tree prediction array shape:
(150, 1000)


In [32]:
print(
    "Number of trees:",
    tree_predictions.shape[0]
)

print(
    "Number of sampled properties:",
    tree_predictions.shape[1]
)

print()

if (
    tree_predictions.shape[1]
    == sample_size
):

    print(
        "✅ Tree prediction dimensions are correct."
    )

else:

    raise ValueError(
        "Tree prediction dimensions are incorrect."
    )

Number of trees: 150
Number of sampled properties: 1000

✅ Tree prediction dimensions are correct.


In [33]:
mean_predictions = np.mean(
    tree_predictions,
    axis=0
)

print(
    "✅ Mean predictions calculated."
)

print()
print(
    "Number of predictions:",
    len(mean_predictions)
)

print()
print(
    "First 10 predictions:"
)

print(
    mean_predictions[:10]
)

✅ Mean predictions calculated.

Number of predictions: 1000

First 10 predictions:
[39804946.76       19180473.33333333 66175745.16222221  2427889.61333333
 18619637.45222222  7343470.26666667 15905923.58888889  4291605.26666667
 16783003.74666667 16340604.31      ]


In [34]:
std_predictions = np.std(
    tree_predictions,
    axis=0
)

print(
    "✅ Prediction standard deviation calculated."
)

print()
print(
    "First 10 standard deviations:"
)

print(
    std_predictions[:10]
)

✅ Prediction standard deviation calculated.

First 10 standard deviations:
[ 7498556.56673407  2848869.07081077 19875572.28461469   750521.20149508
  5416020.72809673  1143592.864009    2843884.82888903   466671.77655889
  2688252.80773045   802376.06055258]


In [35]:
lower_predictions = (
    mean_predictions
    - 1.96 * std_predictions
)

upper_predictions = (
    mean_predictions
    + 1.96 * std_predictions
)

# House prices cannot be negative
lower_predictions = np.maximum(
    lower_predictions,
    0
)

print(
    "✅ Lower and upper price estimates calculated."
)

print()
print(
    "First 5 estimated ranges:"
)

for i in range(
    min(5, len(mean_predictions))
):

    print(
        f"₹{lower_predictions[i]:,.0f}"
        f" - "
        f"₹{upper_predictions[i]:,.0f}"
    )

✅ Lower and upper price estimates calculated.

First 5 estimated ranges:
₹25,107,776 - ₹54,502,118
₹13,596,690 - ₹24,764,257
₹27,219,623 - ₹105,131,867
₹956,868 - ₹3,898,911
₹8,004,237 - ₹29,235,038


In [36]:
confidence_df = sample_df.copy()

confidence_df[
    "confidence_predicted_price"
] = mean_predictions

confidence_df[
    "prediction_std"
] = std_predictions

confidence_df[
    "lower_price_estimate"
] = lower_predictions

confidence_df[
    "upper_price_estimate"
] = upper_predictions

confidence_df[
    "prediction_range"
] = (
    confidence_df[
        "upper_price_estimate"
    ]
    -
    confidence_df[
        "lower_price_estimate"
    ]
)

print(
    "✅ Confidence dataframe created."
)

print()
print(
    "Shape:",
    confidence_df.shape
)

display(
    confidence_df.head()
)

✅ Confidence dataframe created.

Shape: (1000, 22)


,price,area,price_per_sqft,locality,city,property_type,bedroom_num,bathroom_num,balcony_num,furnished,...,latitude,longitude,area_per_bedroom,bathroom_per_bedroom,age_group,confidence_predicted_price,prediction_std,lower_price_estimate,upper_price_estimate,prediction_range
45742,39500000,1612,24503.722084,Powai,Mumbai,Apartment,3,3,0,Unfurnished,...,19.111784,72.898430,537.333333,1.0,New,3.980495e+07,7.498557e+06,2.510778e+07,5.450212e+07,2.939434e+07
6623,16500000,1743,9466.437177,Ulwe,Mumbai,Apartment,3,3,0,Unfurnished,...,18.967709,73.022980,581.000000,1.0,New,1.918047e+07,2.848869e+06,1.359669e+07,2.476426e+07,1.116757e+07
39701,75000000,1250,60000.000000,Breach Candy,Mumbai,Apartment,2,2,2,Furnished,...,18.973936,72.805511,625.000000,1.0,Moderate,6.617575e+07,1.987557e+07,2.721962e+07,1.051319e+08,7.791224e+07
16332,1851000,675,2742.222222,Boisar,Mumbai,Apartment,2,2,0,Unfurnished,...,19.833578,72.718498,337.500000,1.0,Recent,2.427890e+06,7.505212e+05,9.568681e+05,3.898911e+06,2.942043e+06
45933,18000000,700,25714.285714,Chembur,Mumbai,Apartment,2,1,0,Furnished,...,19.048319,72.906250,350.000000,0.5,New,1.861964e+07,5.416021e+06,8.004237e+06,2.923504e+07,2.123080e+07


In [37]:
confidence_df[
    "uncertainty_percent"
] = (
    confidence_df[
        "prediction_std"
    ]
    /
    confidence_df[
        "confidence_predicted_price"
    ]
    * 100
)

confidence_df[
    "uncertainty_percent"
] = (
    confidence_df[
        "uncertainty_percent"
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

print(
    "✅ Uncertainty percentage calculated."
)

display(
    confidence_df[
        [
            "confidence_predicted_price",
            "prediction_std",
            "uncertainty_percent"
        ]
    ].head(10)
)

✅ Uncertainty percentage calculated.


,confidence_predicted_price,prediction_std,uncertainty_percent
45742,3.980495e+07,7.498557e+06,18.838253
6623,1.918047e+07,2.848869e+06,14.852965
39701,6.617575e+07,1.987557e+07,30.034527
16332,2.427890e+06,7.505212e+05,30.912493
45933,1.861964e+07,5.416021e+06,29.087681
17777,7.343470e+06,1.143593e+06,15.572922
19599,1.590592e+07,2.843885e+06,17.879407
42677,4.291605e+06,4.666718e+05,10.874061
41061,1.678300e+07,2.688253e+06,16.017710
24835,1.634060e+07,8.023761e+05,4.910321


In [38]:
def assign_confidence(
    uncertainty
):

    if pd.isna(
        uncertainty
    ):

        return "Unknown"

    elif uncertainty <= 5:

        return "High Confidence"

    elif uncertainty <= 10:

        return "Medium Confidence"

    else:

        return "Low Confidence"


print(
    "✅ Confidence classification function created."
)

✅ Confidence classification function created.


In [39]:
confidence_df[
    "confidence_level"
] = confidence_df[
    "uncertainty_percent"
].apply(
    assign_confidence
)

print(
    "✅ Confidence levels assigned."
)

display(
    confidence_df[
        [
            "confidence_predicted_price",
            "uncertainty_percent",
            "confidence_level"
        ]
    ].head(15)
)

✅ Confidence levels assigned.


,confidence_predicted_price,uncertainty_percent,confidence_level
45742,3.980495e+07,18.838253,Low Confidence
6623,1.918047e+07,14.852965,Low Confidence
39701,6.617575e+07,30.034527,Low Confidence
16332,2.427890e+06,30.912493,Low Confidence
45933,1.861964e+07,29.087681,Low Confidence
17777,7.343470e+06,15.572922,Low Confidence
19599,1.590592e+07,17.879407,Low Confidence
42677,4.291605e+06,10.874061,Low Confidence
41061,1.678300e+07,16.017710,Low Confidence
24835,1.634060e+07,4.910321,High Confidence


In [40]:
confidence_df[
    "estimated_price_range"
] = (
    "₹"
    + confidence_df[
        "lower_price_estimate"
    ]
    .round()
    .astype(int)
    .map(
        lambda x: f"{x:,}"
    )
    + " - ₹"
    + confidence_df[
        "upper_price_estimate"
    ]
    .round()
    .astype(int)
    .map(
        lambda x: f"{x:,}"
    )
)

print(
    "✅ Estimated price ranges created."
)

display(
    confidence_df[
        [
            "confidence_predicted_price",
            "lower_price_estimate",
            "upper_price_estimate",
            "estimated_price_range",
            "uncertainty_percent",
            "confidence_level"
        ]
    ].head(15)
)

✅ Estimated price ranges created.


,confidence_predicted_price,lower_price_estimate,upper_price_estimate,estimated_price_range,uncertainty_percent,confidence_level
45742,3.980495e+07,2.510778e+07,5.450212e+07,"₹25,107,776 - ₹54,502,118",18.838253,Low Confidence
6623,1.918047e+07,1.359669e+07,2.476426e+07,"₹13,596,690 - ₹24,764,257",14.852965,Low Confidence
39701,6.617575e+07,2.721962e+07,1.051319e+08,"₹27,219,623 - ₹105,131,867",30.034527,Low Confidence
16332,2.427890e+06,9.568681e+05,3.898911e+06,"₹956,868 - ₹3,898,911",30.912493,Low Confidence
45933,1.861964e+07,8.004237e+06,2.923504e+07,"₹8,004,237 - ₹29,235,038",29.087681,Low Confidence
17777,7.343470e+06,5.102028e+06,9.584912e+06,"₹5,102,028 - ₹9,584,912",15.572922,Low Confidence
19599,1.590592e+07,1.033191e+07,2.147994e+07,"₹10,331,909 - ₹21,479,938",17.879407,Low Confidence
42677,4.291605e+06,3.376929e+06,5.206282e+06,"₹3,376,929 - ₹5,206,282",10.874061,Low Confidence
41061,1.678300e+07,1.151403e+07,2.205198e+07,"₹11,514,028 - ₹22,051,979",16.017710,Low Confidence
24835,1.634060e+07,1.476795e+07,1.791326e+07,"₹14,767,947 - ₹17,913,261",4.910321,High Confidence


In [41]:
print("=" * 70)
print("📊 PREDICTION CONFIDENCE SUMMARY")
print("=" * 70)

print()

print(
    "Properties analyzed:",
    len(confidence_df)
)

print(
    "Average predicted price: ₹{:,.0f}".format(
        confidence_df[
            "confidence_predicted_price"
        ].mean()
    )
)

print(
    "Average uncertainty: {:.2f}%".format(
        confidence_df[
            "uncertainty_percent"
        ].mean()
    )
)

print(
    "Average prediction range: ₹{:,.0f}".format(
        confidence_df[
            "prediction_range"
        ].mean()
    )
)

📊 PREDICTION CONFIDENCE SUMMARY

Properties analyzed: 1000
Average predicted price: ₹19,899,058
Average uncertainty: 18.32%
Average prediction range: ₹16,970,861


In [42]:
if "predicted_price" in confidence_df.columns:

    confidence_df[
        "prediction_difference"
    ] = (
        confidence_df[
            "confidence_predicted_price"
        ]
        -
        confidence_df[
            "predicted_price"
        ]
    )

    confidence_df[
        "prediction_difference_percent"
    ] = (
        confidence_df[
            "prediction_difference"
        ]
        /
        confidence_df[
            "predicted_price"
        ]
        * 100
    )

    print(
        "✅ Compared with existing prediction."
    )

    display(
        confidence_df[
            [
                "predicted_price",
                "confidence_predicted_price",
                "prediction_difference",
                "prediction_difference_percent"
            ]
        ].head(10)
    )

else:

    print(
        "ℹ️ Existing predicted_price column is not available."
    )

ℹ️ Existing predicted_price column is not available.


In [43]:
confidence_distribution = (
    confidence_df[
        "confidence_level"
    ]
    .value_counts()
    .reset_index()
)

confidence_distribution.columns = [
    "confidence_level",
    "property_count"
]

print(
    "📊 Confidence Level Distribution"
)

display(
    confidence_distribution
)

📊 Confidence Level Distribution


,confidence_level,property_count
0,Low Confidence,802
1,Medium Confidence,147
2,High Confidence,51


In [44]:
uncertainty_statistics = (
    confidence_df[
        "uncertainty_percent"
    ]
    .describe()
    .to_frame(
        "uncertainty_percent"
    )
)

print(
    "📊 Uncertainty Statistics"
)

display(
    uncertainty_statistics
)

📊 Uncertainty Statistics


,uncertainty_percent
count,1000.000000
mean,18.316689
std,11.464120
min,0.316564
25%,10.943830
50%,16.764158
75%,22.779438
max,106.452491


In [45]:
most_certain = (
    confidence_df
    .sort_values(
        "uncertainty_percent",
        ascending=True
    )
    .head(10)
)

print(
    "🟢 Most Certain Predictions"
)

columns_to_display = [
    "locality",
    "area",
    "bedroom_num",
    "confidence_predicted_price",
    "uncertainty_percent",
    "confidence_level",
    "estimated_price_range"
]

available_columns = [
    column
    for column in columns_to_display
    if column in most_certain.columns
]

display(
    most_certain[
        available_columns
    ]
)

🟢 Most Certain Predictions


,locality,area,bedroom_num,confidence_predicted_price,uncertainty_percent,confidence_level,estimated_price_range
42825,Ulwe,425,1,2.219995e+06,0.316564,High Confidence,"₹2,206,221 - ₹2,233,769"
37009,Chembur,555,2,1.444728e+07,0.472277,High Confidence,"₹14,313,550 - ₹14,581,016"
5763,Bandra,1381,3,6.696284e+07,0.598934,High Confidence,"₹66,176,761 - ₹67,748,928"
21051,Nalasopara,531,2,4.297733e+06,0.643786,High Confidence,"₹4,243,504 - ₹4,351,963"
21430,Thane,1350,3,1.898000e+07,0.853899,High Confidence,"₹18,662,343 - ₹19,297,657"
29624,Ulwe,485,2,1.334233e+07,1.013614,High Confidence,"₹13,077,264 - ₹13,607,404"
5762,Bandra,1441,3,7.040202e+07,1.127254,High Confidence,"₹68,846,548 - ₹71,957,498"
32909,Karanjade,281,1,4.322069e+06,1.191758,High Confidence,"₹4,221,112 - ₹4,423,026"
22503,Kandivali,422,1,9.938759e+06,1.208934,High Confidence,"₹9,703,259 - ₹10,174,259"
32329,Kharghar,582,2,1.203809e+07,1.686174,High Confidence,"₹11,640,247 - ₹12,435,941"


In [46]:
least_certain = (
    confidence_df
    .sort_values(
        "uncertainty_percent",
        ascending=False
    )
    .head(10)
)

print(
    "🔴 Least Certain Predictions"
)

available_columns = [
    column
    for column in columns_to_display
    if column in least_certain.columns
]

display(
    least_certain[
        available_columns
    ]
)

🔴 Least Certain Predictions


,locality,area,bedroom_num,confidence_predicted_price,uncertainty_percent,confidence_level,estimated_price_range
46216,Kalyan,720,1,7.478972e+06,106.452491,Low Confidence,"₹0 - ₹23,083,613"
51439,Andheri,2187,4,8.817233e+07,101.627241,Low Confidence,"₹0 - ₹263,802,255"
21669,Manpada,6000,4,7.618843e+07,85.109865,Low Confidence,"₹0 - ₹203,282,408"
39372,Borivali,225,1,9.750547e+06,83.397925,Low Confidence,"₹0 - ₹25,688,785"
29740,Borivali,741,2,2.045612e+07,82.382329,Low Confidence,"₹0 - ₹53,486,479"
23280,Chembur,4387,4,7.084982e+07,78.320592,Low Confidence,"₹0 - ₹179,610,223"
40877,Bhiwandi,585,1,3.199285e+06,70.182113,Low Confidence,"₹0 - ₹7,600,124"
6225,Andheri,1700,3,2.015956e+07,66.275296,Low Confidence,"₹0 - ₹46,346,744"
28334,Mira Road,2250,5,2.872983e+07,66.258226,Low Confidence,"₹0 - ₹66,040,136"
11667,Karjat,2758,2,1.365638e+07,65.199707,Low Confidence,"₹0 - ₹31,108,060"


In [47]:
## 🏙️ Locality-wise Prediction Uncertainty

The next analysis examines prediction uncertainty across Mumbai localities.

This helps identify localities where the Random Forest produces relatively
consistent predictions and localities where the model shows greater variation.

SyntaxError: invalid syntax (690004406.py, line 3)

In [48]:
if "locality" not in confidence_df.columns:

    raise ValueError(
        "The locality column is not available in confidence_df."
    )


locality_uncertainty = (
    confidence_df
    .groupby("locality")
    .agg(
        property_count=(
            "confidence_predicted_price",
            "count"
        ),
        average_predicted_price=(
            "confidence_predicted_price",
            "mean"
        ),
        average_uncertainty=(
            "uncertainty_percent",
            "mean"
        ),
        average_prediction_std=(
            "prediction_std",
            "mean"
        ),
        average_prediction_range=(
            "prediction_range",
            "mean"
        )
    )
    .reset_index()
)

locality_uncertainty = (
    locality_uncertainty
    .sort_values(
        "average_uncertainty",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "✅ Locality-wise uncertainty analysis created."
)

print()
print(
    "Number of localities:",
    len(locality_uncertainty)
)

display(
    locality_uncertainty.head(20)
)

✅ Locality-wise uncertainty analysis created.

Number of localities: 102


,locality,property_count,average_predicted_price,average_uncertainty,average_prediction_std,average_prediction_range
0,Rabale Station Road,1,3.906667e+06,60.895803,2.378996e+06,8.569499e+06
1,Manpada,2,4.694852e+07,52.977656,3.426765e+07,1.088764e+08
2,Tardeo,1,1.873950e+08,45.761740,8.575523e+07,3.361605e+08
3,Girgaon,2,3.002968e+07,41.340164,1.336500e+07,4.811942e+07
4,Karjat,5,7.982683e+06,38.934114,3.500163e+06,1.296158e+07
5,Cuffe Parade,1,1.354015e+08,38.077137,5.155700e+07,2.021035e+08
6,Worli,3,9.441473e+07,36.922202,3.830190e+07,1.499743e+08
7,Marine Lines,1,1.634862e+07,36.472938,5.962823e+06,2.337426e+07
8,Sai Samarth Mitra,2,6.162833e+06,36.094819,2.223652e+06,8.716716e+06
9,Umroli,1,4.138963e+06,35.277825,1.460136e+06,5.723733e+06


In [49]:
highest_uncertainty_localities = (
    locality_uncertainty
    .sort_values(
        "average_uncertainty",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

print(
    "🔴 Localities with Highest Prediction Uncertainty"
)

display(
    highest_uncertainty_localities
)

🔴 Localities with Highest Prediction Uncertainty


,locality,property_count,average_predicted_price,average_uncertainty,average_prediction_std,average_prediction_range
0,Rabale Station Road,1,3.906667e+06,60.895803,2.378996e+06,8.569499e+06
1,Manpada,2,4.694852e+07,52.977656,3.426765e+07,1.088764e+08
2,Tardeo,1,1.873950e+08,45.761740,8.575523e+07,3.361605e+08
3,Girgaon,2,3.002968e+07,41.340164,1.336500e+07,4.811942e+07
4,Karjat,5,7.982683e+06,38.934114,3.500163e+06,1.296158e+07
5,Cuffe Parade,1,1.354015e+08,38.077137,5.155700e+07,2.021035e+08
6,Worli,3,9.441473e+07,36.922202,3.830190e+07,1.499743e+08
7,Marine Lines,1,1.634862e+07,36.472938,5.962823e+06,2.337426e+07
8,Sai Samarth Mitra,2,6.162833e+06,36.094819,2.223652e+06,8.716716e+06
9,Umroli,1,4.138963e+06,35.277825,1.460136e+06,5.723733e+06


In [50]:
lowest_uncertainty_localities = (
    locality_uncertainty
    .sort_values(
        "average_uncertainty",
        ascending=True
    )
    .head(10)
    .reset_index(drop=True)
)

print(
    "🟢 Localities with Lowest Prediction Uncertainty"
)

display(
    lowest_uncertainty_localities
)

🟢 Localities with Lowest Prediction Uncertainty


,locality,property_count,average_predicted_price,average_uncertainty,average_prediction_std,average_prediction_range
0,Nalasopara,1,4.297733e+06,0.643786,2.766819e+04,1.084593e+05
1,Anjurdive,1,6.944325e+06,7.020551,4.875299e+05,1.911117e+06
2,Titwala,1,2.672971e+06,8.233700,2.200844e+05,8.627309e+05
3,Bhayandar,10,7.658602e+06,8.764791,6.508569e+05,2.551359e+06
4,Deonar,2,2.440210e+07,9.170766,2.221152e+06,8.706916e+06
5,Naigaon,20,4.980529e+06,10.476662,4.956108e+05,1.942794e+06
6,Ambernath,18,3.089785e+06,10.927131,3.681065e+05,1.442977e+06
7,Ambarnath,1,2.835672e+06,10.986488,3.115407e+05,1.221240e+06
8,Mira Road,51,8.779493e+06,11.420735,1.236049e+06,4.677069e+06
9,Kurla,10,1.267239e+07,12.386309,1.529278e+06,5.994772e+06


In [51]:
prediction_range_summary = (
    confidence_df[
        [
            "confidence_predicted_price",
            "lower_price_estimate",
            "upper_price_estimate",
            "prediction_range",
            "uncertainty_percent"
        ]
    ]
    .describe()
)

print(
    "📊 Prediction Range Summary"
)

display(
    prediction_range_summary
)

📊 Prediction Range Summary


,confidence_predicted_price,lower_price_estimate,upper_price_estimate,prediction_range,uncertainty_percent
count,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1000.000000
mean,1.989906e+07,1.153523e+07,2.850609e+07,1.697086e+07,18.316689
std,2.672528e+07,1.292254e+07,4.497591e+07,3.829864e+07,11.464120
min,1.594822e+06,0.000000e+00,1.993792e+06,2.754860e+04,0.316564
25%,6.983284e+06,4.421814e+06,9.092864e+06,3.395485e+06,10.943830
50%,1.255373e+07,7.854769e+06,1.643200e+07,7.758913e+06,16.764158
75%,2.171001e+07,1.387668e+07,2.976322e+07,1.621653e+07,22.779438
max,3.049273e+08,1.585170e+08,5.951747e+08,5.911790e+08,106.452491


In [52]:
confidence_output_path = os.path.join(
    OUTPUT_DIR,
    "prediction_confidence_results.csv"
)

confidence_df.to_csv(
    confidence_output_path,
    index=False
)

print(
    "✅ Prediction confidence results exported."
)

print()
print(
    confidence_output_path
)

✅ Prediction confidence results exported.

C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\outputs\prediction_confidence_results.csv


In [53]:
locality_output_path = os.path.join(
    OUTPUT_DIR,
    "locality_prediction_uncertainty.csv"
)

locality_uncertainty.to_csv(
    locality_output_path,
    index=False
)

print(
    "✅ Locality uncertainty results exported."
)

print()
print(
    locality_output_path
)

✅ Locality uncertainty results exported.

C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\outputs\locality_prediction_uncertainty.csv


In [54]:
print("=" * 70)
print("📁 VERIFYING OUTPUT FILES")
print("=" * 70)

print()

if os.path.exists(
    confidence_output_path
):

    print(
        "✅ prediction_confidence_results.csv"
    )

else:

    print(
        "❌ prediction_confidence_results.csv"
    )


if os.path.exists(
    locality_output_path
):

    print(
        "✅ locality_prediction_uncertainty.csv"
    )

else:

    print(
        "❌ locality_prediction_uncertainty.csv"
    )

📁 VERIFYING OUTPUT FILES

✅ prediction_confidence_results.csv
✅ locality_prediction_uncertainty.csv


In [55]:
final_columns = [
    "locality",
    "property_type",
    "area",
    "bedroom_num",
    "bathroom_num",
    "confidence_predicted_price",
    "lower_price_estimate",
    "upper_price_estimate",
    "estimated_price_range",
    "uncertainty_percent",
    "confidence_level"
]

available_final_columns = [
    column
    for column in final_columns
    if column in confidence_df.columns
]

final_confidence_view = confidence_df[
    available_final_columns
].copy()

print(
    "🏠 FINAL PROPERTY CONFIDENCE RESULTS"
)

display(
    final_confidence_view.head(20)
)

🏠 FINAL PROPERTY CONFIDENCE RESULTS


,locality,property_type,area,bedroom_num,bathroom_num,confidence_predicted_price,lower_price_estimate,upper_price_estimate,estimated_price_range,uncertainty_percent,confidence_level
45742,Powai,Apartment,1612,3,3,3.980495e+07,2.510778e+07,5.450212e+07,"₹25,107,776 - ₹54,502,118",18.838253,Low Confidence
6623,Ulwe,Apartment,1743,3,3,1.918047e+07,1.359669e+07,2.476426e+07,"₹13,596,690 - ₹24,764,257",14.852965,Low Confidence
39701,Breach Candy,Apartment,1250,2,2,6.617575e+07,2.721962e+07,1.051319e+08,"₹27,219,623 - ₹105,131,867",30.034527,Low Confidence
16332,Boisar,Apartment,675,2,2,2.427890e+06,9.568681e+05,3.898911e+06,"₹956,868 - ₹3,898,911",30.912493,Low Confidence
45933,Chembur,Apartment,700,2,1,1.861964e+07,8.004237e+06,2.923504e+07,"₹8,004,237 - ₹29,235,038",29.087681,Low Confidence
17777,Kalyan,Apartment,950,2,2,7.343470e+06,5.102028e+06,9.584912e+06,"₹5,102,028 - ₹9,584,912",15.572922,Low Confidence
19599,Koper Khairane,Apartment,1060,2,2,1.590592e+07,1.033191e+07,2.147994e+07,"₹10,331,909 - ₹21,479,938",17.879407,Low Confidence
42677,Dombivli,Apartment,630,1,1,4.291605e+06,3.376929e+06,5.206282e+06,"₹3,376,929 - ₹5,206,282",10.874061,Low Confidence
41061,Kurla,Apartment,750,2,2,1.678300e+07,1.151403e+07,2.205198e+07,"₹11,514,028 - ₹22,051,979",16.017710,Low Confidence
24835,Malad,Apartment,719,2,2,1.634060e+07,1.476795e+07,1.791326e+07,"₹14,767,947 - ₹17,913,261",4.910321,High Confidence


In [56]:
print("=" * 70)
print("📊 PREDICTION CONFIDENCE ANALYSIS — FINAL SUMMARY")
print("=" * 70)

print()

print(
    "Total properties analyzed:",
    len(confidence_df)
)

print()

print(
    "Average predicted price: ₹{:,.0f}".format(
        confidence_df[
            "confidence_predicted_price"
        ].mean()
    )
)

print(
    "Average uncertainty: {:.2f}%".format(
        confidence_df[
            "uncertainty_percent"
        ].mean()
    )
)

print(
    "Average prediction range: ₹{:,.0f}".format(
        confidence_df[
            "prediction_range"
        ].mean()
    )
)

print()

print(
    "Confidence distribution:"
)

print(
    confidence_df[
        "confidence_level"
    ].value_counts()
)

print()

print(
    "Generated files:"
)

print(
    "1. prediction_confidence_results.csv"
)

print(
    "2. locality_prediction_uncertainty.csv"
)

print()

print("=" * 70)
print("✅ NOTEBOOK COMPLETED SUCCESSFULLY")
print("=" * 70)

📊 PREDICTION CONFIDENCE ANALYSIS — FINAL SUMMARY

Total properties analyzed: 1000

Average predicted price: ₹19,899,058
Average uncertainty: 18.32%
Average prediction range: ₹16,970,861

Confidence distribution:
confidence_level
Low Confidence       802
Medium Confidence    147
High Confidence       51
Name: count, dtype: int64

Generated files:
1. prediction_confidence_results.csv
2. locality_prediction_uncertainty.csv

✅ NOTEBOOK COMPLETED SUCCESSFULLY


In [57]:
# ✅ Prediction Confidence Analysis Completed

The prediction confidence module has been successfully completed.

### Analysis Generated

- Random Forest individual tree predictions
- Mean predicted property price
- Prediction standard deviation
- Lower estimated price
- Upper estimated price
- Estimated price range
- Prediction uncertainty percentage
- Confidence level
- Most certain properties
- Least certain properties
- Locality-wise uncertainty analysis

### Generated Files

- `prediction_confidence_results.csv`
- `locality_prediction_uncertainty.csv`

### Confidence Categories

The project uses the following model-based categories:

- **High Confidence:** uncertainty <= 5%
- **Medium Confidence:** uncertainty > 5% and <= 10%
- **Low Confidence:** uncertainty > 10%

These thresholds are project-defined and are not externally validated statistical
confidence thresholds.

### Important Limitation

The estimated price range is calculated from the variation among individual
Random Forest trees.

It should therefore be interpreted as a model-based uncertainty estimate and
NOT as a formal statistical confidence interval.

### Project Integration

The confidence results can later be integrated into the Flask dashboard so
that users can see:

**Predicted Price + Estimated Price Range + Confidence Level**

SyntaxError: invalid syntax (4060613590.py, line 3)